# 🎯 InterviewIQ AI — Answer Quality & Score Prediction
### Notebook 03: Supervised Model Training for Answer Evaluation

This notebook trains supervised machine learning models to score candidate interview answers:
1. **Answer Scorer (Regression)**: Predicts continuous quality score $[0, 100]$.
2. **Quality Classifier (Classification)**: Categorizes response into `poor`, `average`, `good`, or `excellent`.

---
#### Approach:
We combine the interview question with the candidate's answer using a separator token `[SEP]`, extract $n$-gram TF-IDF representations, and train **Ridge Regression ($L_2$ regularization)** and **Multinomial Logistic Regression**.

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, classification_report

data_path = '../data/interview_answers.csv'
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} answer samples.")
df.head(2)

## 1. Feature Engineering & Target Scaling
We scale the target score from $[1, 10]$ to a standard percentage scale $[10, 100]$.

In [ ]:
df['combined_input'] = df['question'] + ' [SEP] ' + df['answer']
df['target_score'] = df['score'] * 10.0

X_train_raw, X_test_raw, y_score_train, y_score_test, y_qual_train, y_qual_test = train_test_split(
    df['combined_input'],
    df['target_score'],
    df['quality_label'],
    test_size=0.20,
    random_state=42,
    stratify=df['quality_label']
)

print(f"Train set: {len(X_train_raw)}, Test set: {len(X_test_raw)}")

## 2. Text Representation with TF-IDF Vectorizer

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    stop_words='english',
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print(f"Extracted {X_train.shape[1]} features.")

## 3. Training Answer Score Regressor (Ridge Regression)

In [ ]:
scorer = Ridge(alpha=1.0, random_state=42)
scorer.fit(X_train, y_score_train)

y_score_pred = scorer.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_score_test, y_score_pred))
mae = mean_absolute_error(y_score_test, y_score_pred)
r2 = r2_score(y_score_test, y_score_pred)

print(f"Ridge Regression Performance:")
print(f"  R² Score:  {r2:.4f}")
print(f"  RMSE:      {rmse:.2f} points (on 0-100 scale)")
print(f"  MAE:       {mae:.2f} points")

## 4. Training Answer Quality Classifier (Logistic Regression)

In [ ]:
classifier = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
classifier.fit(X_train, y_qual_train)

y_qual_pred = classifier.predict(X_test)
print("Classification Report for Answer Quality:")
print(classification_report(y_qual_test, y_qual_pred))

## 5. Exporting Answer Evaluation Models

In [ ]:
joblib.dump(scorer, os.path.join(models_dir, 'answer_scorer.joblib'))
joblib.dump(vectorizer, os.path.join(models_dir, 'answer_vectorizer.joblib'))
joblib.dump(classifier, os.path.join(models_dir, 'answer_quality_classifier.joblib'))

print("All answer scoring artifacts saved to ../models/")